In [ ]:
# ارزیابی های مدل
# BaseVAE = Vanilla-AE
# Beta-VAE
# HierarchicalVAE
# FactorVAE
# LadderVAE

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

########################################
# CPU Optimization
########################################

os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"

torch.set_num_threads(8)
torch.set_num_interop_threads(8)

device = torch.device("cpu")

class MelDataset(Dataset):

    def __init__(self, root):
        self.files = [
            os.path.join(root, f)
            for f in os.listdir(root)
            if f.endswith(".pt")
        ]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        x = torch.load(self.files[idx], weights_only=True)

        # force fixed size 80x80
        if x.shape[2] > 80:
            x = x[:, :, :80]
        else:
            pad = 80 - x.shape[2]
            x = F.pad(x, (0, pad))

        return x


train_dataset = MelDataset("data/librispeech_mel/train-clean-100")
test_dataset  = MelDataset("data/librispeech_mel/dev-clean")

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=6
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=6
)


In [2]:
x = next(iter(train_loader))
print(x.shape)


torch.Size([32, 1, 80, 80])


In [3]:

vanilla_model = BaseVAE(
    latent_dim=120,
    input_shape=(1,80,80)
).to(device)


recon, mu, logvar = vanilla_model(x)

print(recon.shape)
print(mu.shape)


NameError: name 'BaseVAE' is not defined

In [12]:

def vae_loss(recon_x, x, mu, logvar, beta=1.0):

    recon_loss = F.mse_loss(recon_x, x)

    kld = -0.5 * torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp()
    )
    # print('kld= ',kld)
    # print(' recon_loss + beta * kld= ', recon_loss + beta * kld)

    return recon_loss + beta * kld, recon_loss, kld

################################
def train_baseline(
    model,
    train_loader,
    device,
    epochs=30,
    lr=1e-4,
    beta=1.0,
    model_name="model"
):

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_loss = float("inf")

    save_path = f"{model_name}_best.pt"

    model.to(device)

    for epoch in range(epochs):
        print('epoch= ',epoch)

        model.train()

        total_loss = 0
        total_rec = 0
        total_kld = 0

        for x in train_loader:

            x = x.to(device)

            recon, mu, logvar = model(x)

            loss, rec, kld = vae_loss(recon, x, mu, logvar, beta)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_rec += rec.item()
            total_kld += kld.item()

        n = len(train_loader)

        epoch_loss = total_loss / n

        print(
            f"{model_name} | Epoch {epoch} | "
            f"Loss {epoch_loss:.4f} | "
            f"Rec {total_rec/n:.4f} | "
            f"KLD {total_kld/n:.4f}"
        )

        if epoch_loss < best_loss:

            best_loss = epoch_loss

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "loss": epoch_loss,
                },
                save_path,
            )

            print("checkpoint saved ✅")

##########################
vanilla_model = BaseVAE(
    latent_dim=120,
    input_shape=(1,80,80)
).to(device)

train_baseline(
    vanilla_model,
    train_loader,
    device,
    epochs=30,
    beta=0,
    model_name="vanilla_ae"
)

#########################################
beta_model = BaseVAE(
    latent_dim=120,
    input_shape=(1,80,80)
).to(device)

train_baseline(
    beta_model,
    train_loader,
    device,
    epochs=30,
    beta=4,
    model_name="beta_vae"
)

############################
hvae_model = HierarchicalVAE()

train_baseline(
    hvae_model,
    train_loader,
    device,
    epochs=30,
    beta=1,
    model_name="hvae"
)


epoch=  0
vanilla_ae | Epoch 0 | Loss 0.8716 | Rec 0.8716 | KLD 858.7869
checkpoint saved ✅
epoch=  1
vanilla_ae | Epoch 1 | Loss 0.7398 | Rec 0.7398 | KLD 363.5989
checkpoint saved ✅
epoch=  2
vanilla_ae | Epoch 2 | Loss 0.7118 | Rec 0.7118 | KLD 251.9259
checkpoint saved ✅
epoch=  3
vanilla_ae | Epoch 3 | Loss 0.6995 | Rec 0.6995 | KLD 195.9146
checkpoint saved ✅
epoch=  4
vanilla_ae | Epoch 4 | Loss 0.6929 | Rec 0.6929 | KLD 160.1604
checkpoint saved ✅
epoch=  5
vanilla_ae | Epoch 5 | Loss 0.6886 | Rec 0.6886 | KLD 139.7359
checkpoint saved ✅
epoch=  6
vanilla_ae | Epoch 6 | Loss 0.6847 | Rec 0.6847 | KLD 127.9560
checkpoint saved ✅
epoch=  7
vanilla_ae | Epoch 7 | Loss 0.6814 | Rec 0.6814 | KLD 119.1589
checkpoint saved ✅
epoch=  8
vanilla_ae | Epoch 8 | Loss 0.6788 | Rec 0.6788 | KLD 112.6372
checkpoint saved ✅
epoch=  9
vanilla_ae | Epoch 9 | Loss 0.6766 | Rec 0.6766 | KLD 108.1051
checkpoint saved ✅
epoch=  10
vanilla_ae | Epoch 10 | Loss 0.6748 | Rec 0.6748 | KLD 104.0927
check

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x51200 and 131072x40)

In [16]:
hvae_model = HierarchicalVAE()

train_baseline(
    hvae_model,
    train_loader,
    device,
    epochs=30,
    beta=1,
    model_name="hvae"
)


epoch=  0
hvae | Epoch 0 | Loss 1.0438 | Rec 1.0274 | KLD 0.0164
checkpoint saved ✅
epoch=  1
hvae | Epoch 1 | Loss 0.9313 | Rec 0.8988 | KLD 0.0325
checkpoint saved ✅
epoch=  2
hvae | Epoch 2 | Loss 0.9154 | Rec 0.8780 | KLD 0.0375
checkpoint saved ✅
epoch=  3
hvae | Epoch 3 | Loss 0.9114 | Rec 0.8729 | KLD 0.0385
checkpoint saved ✅
epoch=  4
hvae | Epoch 4 | Loss 0.9087 | Rec 0.8692 | KLD 0.0395
checkpoint saved ✅
epoch=  5
hvae | Epoch 5 | Loss 0.9062 | Rec 0.8654 | KLD 0.0407
checkpoint saved ✅
epoch=  6
hvae | Epoch 6 | Loss 0.9024 | Rec 0.8608 | KLD 0.0416
checkpoint saved ✅
epoch=  7
hvae | Epoch 7 | Loss 0.8995 | Rec 0.8568 | KLD 0.0427
checkpoint saved ✅
epoch=  8
hvae | Epoch 8 | Loss 0.8977 | Rec 0.8540 | KLD 0.0437
checkpoint saved ✅
epoch=  9
hvae | Epoch 9 | Loss 0.8959 | Rec 0.8511 | KLD 0.0448
checkpoint saved ✅
epoch=  10
hvae | Epoch 10 | Loss 0.8946 | Rec 0.8484 | KLD 0.0462
checkpoint saved ✅
epoch=  11
hvae | Epoch 11 | Loss 0.8928 | Rec 0.8455 | KLD 0.0473
checkpo

In [4]:
####### FactorVAE model
import torch
import torch.nn as nn
import torch.nn.functional as F


class FactorVAE(nn.Module):

    def __init__(self, latent_dim=120, input_shape=(1,80,80)):
        super().__init__()

        self.latent_dim = latent_dim

        ################################
        # Encoder
        ################################

        self.encoder = nn.Sequential(

            nn.Conv2d(1,32,3,2,1), nn.ReLU(),
            nn.Conv2d(32,64,3,2,1), nn.ReLU(),
            nn.Conv2d(64,128,3,2,1), nn.ReLU(),
            nn.Conv2d(128,256,3,2,1), nn.ReLU()
        )

        with torch.no_grad():
            dummy = torch.zeros(1,*input_shape)
            h = self.encoder(dummy)
            self.enc_shape = h.shape[1:]
            self.flatten_dim = h.view(1,-1).shape[1]

        ################################
        # Latent
        ################################

        self.fc_mu = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_var = nn.Linear(self.flatten_dim, latent_dim)

        ################################
        # Decoder
        ################################

        self.decoder_input = nn.Linear(latent_dim,self.flatten_dim)

        self.decoder = nn.Sequential(

            nn.ConvTranspose2d(256,128,3,2,1,1),
            nn.ReLU(),

            nn.ConvTranspose2d(128,64,3,2,1,1),
            nn.ReLU(),

            nn.ConvTranspose2d(64,32,3,2,1,1),
            nn.ReLU(),

            nn.ConvTranspose2d(32,1,3,2,1,1),
            nn.Sigmoid()
        )


    def encode(self,x):

        h = self.encoder(x)

        h_flat = h.view(x.size(0),-1)

        mu = self.fc_mu(h_flat)
        logvar = self.fc_var(h_flat)

        return mu,logvar


    def reparameterize(self,mu,logvar):

        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)

        return mu + eps*std


    def decode(self,z):

        h = self.decoder_input(z)

        h = h.view(z.size(0),*self.enc_shape)

        return self.decoder(h)


    def forward(self,x):

        mu,logvar = self.encode(x)

        z = self.reparameterize(mu,logvar)

        recon = self.decode(z)

        return recon,mu,logvar,z
#############
class FactorVAEDiscriminator(nn.Module):

    def __init__(self,latent_dim=120):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(latent_dim,1000),
            nn.LeakyReLU(0.2),

            nn.Linear(1000,1000),
            nn.LeakyReLU(0.2),

            nn.Linear(1000,1000),
            nn.LeakyReLU(0.2),

            nn.Linear(1000,2)
        )

    def forward(self,z):

        return self.net(z)
###
# Permutation trick
# برای تخمین Total Correlation
def permute_dims(z):

    B, D = z.size()

    perm_z = []

    for d in range(D):

        perm = torch.randperm(B).to(z.device)

        perm_z.append(z[perm,d])

    perm_z = torch.stack(perm_z,dim=1)

    return perm_z


def factor_vae_loss(recon,x,mu,logvar,z,D,gamma=10):

    ################################
    # reconstruction
    ################################

    recon_loss = F.mse_loss(recon,x)

    ################################
    # KL
    ################################

    kld = -0.5*torch.mean(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    ################################
    # Total Correlation
    ################################

    logits = D(z)

    tc = (logits[:,0] - logits[:,1]).mean()

    ################################

    loss = recon_loss + kld + gamma*tc

    return loss,recon_loss,kld,tc


In [25]:
def train_factorvae(
    model,
    discriminator,
    train_loader,
    device,
    epochs=30,
    lr=1e-4,
    gamma=10,
    save_path="factorvae_best.pt"
):

    model.to(device)
    discriminator.to(device)

    opt_vae = torch.optim.Adam(model.parameters(), lr=lr)
    opt_d   = torch.optim.Adam(discriminator.parameters(), lr=lr)

    best_loss = float("inf")

    for epoch in range(epochs):

        print('epoch= ',epoch)
        model.train()
        discriminator.train()

        epoch_loss = 0

        for x in train_loader:

            x = x.to(device)

            ################################
            # ---- VAE forward & update ----
            ################################

            recon, mu, logvar, z = model(x)

            loss, recon_loss, kld, tc = factor_vae_loss(
                recon, x, mu, logvar, z, discriminator, gamma
            )

            opt_vae.zero_grad()
            loss.backward(retain_graph=True)
            opt_vae.step()

            ################################
            # ---- Discriminator update ----
            ################################

            z_perm = permute_dims(z).detach()

            D_real = discriminator(z.detach())
            D_perm = discriminator(z_perm)

            labels_real = torch.zeros(z.size(0), dtype=torch.long).to(device)
            labels_perm = torch.ones(z.size(0), dtype=torch.long).to(device)

            d_loss = (
                F.cross_entropy(D_real, labels_real) +
                F.cross_entropy(D_perm, labels_perm)
            )

            opt_d.zero_grad()
            d_loss.backward()
            opt_d.step()

            epoch_loss += loss.item()

        epoch_loss /= len(train_loader)

        print(
            f"[Epoch {epoch}] "
            f"Loss {epoch_loss:.4f} | "
            f"Recon {recon_loss:.4f} | "
            f"KLD {kld:.4f} | "
            f"TC {tc:.4f}"
        )

        ################################
        # ---- Best checkpoint ----
        ################################

        if epoch_loss < best_loss:

            best_loss = epoch_loss

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "discriminator_state_dict": discriminator.state_dict(),
                    "optimizer_vae_state_dict": opt_vae.state_dict(),
                    "optimizer_d_state_dict": opt_d.state_dict(),
                    "loss": epoch_loss,
                },
                save_path,
            )

            print("✅ FactorVAE checkpoint saved")


In [26]:
factor_model = FactorVAE(
    latent_dim=120,
    input_shape=(1,80,80)
)

discriminator = FactorVAEDiscriminator(
    latent_dim=120
)

train_factorvae(

    factor_model,
    discriminator,
    train_loader,
    device,
    epochs=30,
    gamma=10
)


epoch=  0
[Epoch 0] Loss 2.1460 | Recon 0.9209 | KLD 1.2741 | TC 0.0007
✅ FactorVAE checkpoint saved
epoch=  1
[Epoch 1] Loss 2.1594 | Recon 0.9020 | KLD 1.2242 | TC -0.0033
epoch=  2
[Epoch 2] Loss 2.1327 | Recon 0.9624 | KLD 1.1409 | TC 0.0080
✅ FactorVAE checkpoint saved
epoch=  3
[Epoch 3] Loss 2.1182 | Recon 0.9564 | KLD 1.1841 | TC -0.0049
✅ FactorVAE checkpoint saved
epoch=  4
[Epoch 4] Loss 2.1564 | Recon 0.9973 | KLD 1.1286 | TC -0.0079
epoch=  5
[Epoch 5] Loss 2.1033 | Recon 1.0099 | KLD 0.9338 | TC 0.0033
✅ FactorVAE checkpoint saved
epoch=  6
[Epoch 6] Loss 2.0650 | Recon 0.9177 | KLD 1.0659 | TC 0.0014
✅ FactorVAE checkpoint saved
epoch=  7
[Epoch 7] Loss 2.0654 | Recon 0.9358 | KLD 1.1455 | TC -0.0085
epoch=  8
[Epoch 8] Loss 2.0937 | Recon 0.9589 | KLD 1.0684 | TC 0.0055
epoch=  9
[Epoch 9] Loss 2.0418 | Recon 0.9479 | KLD 1.0520 | TC 0.0047
✅ FactorVAE checkpoint saved
epoch=  10
[Epoch 10] Loss 1.9904 | Recon 0.9612 | KLD 0.9738 | TC 0.0057
✅ FactorVAE checkpoint saved

In [9]:
# Ladder Model
import torch
import torch.nn as nn
import torch.nn.functional as F


class LadderVAE(nn.Module):

    def __init__(self, latent_dims=[40,40,40]):
        super().__init__()

        z1,z2,z3 = latent_dims

        ################################
        # Bottom-up encoder
        ################################

        self.enc1 = nn.Conv2d(1,32,3,stride=2,padding=1)
        self.enc2 = nn.Conv2d(32,64,3,stride=2,padding=1)
        self.enc3 = nn.Conv2d(64,128,3,stride=2,padding=1)

        ################################
        # q(z|x)
        ################################

        self.q_mu1 = nn.Linear(32*40*40, z1)
        self.q_var1 = nn.Linear(32*40*40, z1)

        self.q_mu2 = nn.Linear(64*20*20, z2)
        self.q_var2 = nn.Linear(64*20*20, z2)

        self.q_mu3 = nn.Linear(128*10*10, z3)
        self.q_var3 = nn.Linear(128*10*10, z3)

        ################################
        # Top-down prior p(z_l | z_{l+1})
        ################################

        self.p_mu2 = nn.Linear(z3, z2)
        self.p_var2 = nn.Linear(z3, z2)

        self.p_mu1 = nn.Linear(z2, z1)
        self.p_var1 = nn.Linear(z2, z1)

        ################################
        # Decoder
        ################################

        self.fc_decode = nn.Linear(z1+z2+z3, 128*10*10)

        self.decoder = nn.Sequential(

            nn.Unflatten(1,(128,10,10)),

            nn.ConvTranspose2d(128,64,3,2,1,1),
            nn.ReLU(),

            nn.ConvTranspose2d(64,32,3,2,1,1),
            nn.ReLU(),

            nn.ConvTranspose2d(32,1,3,2,1,1),
            nn.Sigmoid()
        )


    def reparameterize(self, mu, logvar):

        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)

        return mu + eps*std


    def combine_gaussians(self, mu_q, logvar_q, mu_p, logvar_p):

        var_q = torch.exp(logvar_q)
        var_p = torch.exp(logvar_p)

        var = 1/(1/var_q + 1/var_p)

        mu = var*(mu_q/var_q + mu_p/var_p)

        logvar = torch.log(var)

        return mu, logvar


    def forward(self,x):

        ################################
        # bottom-up
        ################################

        h1 = torch.relu(self.enc1(x))
        h2 = torch.relu(self.enc2(h1))
        h3 = torch.relu(self.enc3(h2))

        q_mu1 = self.q_mu1(h1.view(x.size(0),-1))
        q_var1 = self.q_var1(h1.view(x.size(0),-1))

        q_mu2 = self.q_mu2(h2.view(x.size(0),-1))
        q_var2 = self.q_var2(h2.view(x.size(0),-1))

        q_mu3 = self.q_mu3(h3.view(x.size(0),-1))
        q_var3 = self.q_var3(h3.view(x.size(0),-1))

        ################################
        # top latent
        ################################

        z3 = self.reparameterize(q_mu3,q_var3)

        ################################
        # ladder step z2
        ################################

        p_mu2 = self.p_mu2(z3)
        p_var2 = self.p_var2(z3)

        mu2,logvar2 = self.combine_gaussians(
            q_mu2,q_var2,p_mu2,p_var2
        )

        z2 = self.reparameterize(mu2,logvar2)

        ################################
        # ladder step z1
        ################################

        p_mu1 = self.p_mu1(z2)
        p_var1 = self.p_var1(z2)

        mu1,logvar1 = self.combine_gaussians(
            q_mu1,q_var1,p_mu1,p_var1
        )

        z1 = self.reparameterize(mu1,logvar1)

        ################################
        # decoder
        ################################

        z = torch.cat([z1,z2,z3],dim=1)

        h = self.fc_decode(z)

        recon = self.decoder(h)

        ################################

        mu = torch.cat([mu1,mu2,q_mu3],dim=1)
        logvar = torch.cat([logvar1,logvar2,q_var3],dim=1)

        return recon, mu, logvar


In [28]:
ladder_model = LadderVAE().to(device)

train_baseline(
    ladder_model,
    train_loader,
    device,
    epochs=30,
    beta=1,
    model_name="ladder_vae"
)


epoch=  0
ladder_vae | Epoch 0 | Loss 1.0675 | Rec 0.9937 | KLD 0.0738
checkpoint saved ✅
epoch=  1
ladder_vae | Epoch 1 | Loss 0.9751 | Rec 0.9251 | KLD 0.0500
checkpoint saved ✅
epoch=  2
ladder_vae | Epoch 2 | Loss 0.9425 | Rec 0.9013 | KLD 0.0412
checkpoint saved ✅
epoch=  3
ladder_vae | Epoch 3 | Loss 0.9219 | Rec 0.8833 | KLD 0.0386
checkpoint saved ✅
epoch=  4
ladder_vae | Epoch 4 | Loss 0.9106 | Rec 0.8715 | KLD 0.0391
checkpoint saved ✅
epoch=  5
ladder_vae | Epoch 5 | Loss 0.9035 | Rec 0.8620 | KLD 0.0414
checkpoint saved ✅
epoch=  6
ladder_vae | Epoch 6 | Loss 0.8986 | Rec 0.8545 | KLD 0.0441
checkpoint saved ✅
epoch=  7
ladder_vae | Epoch 7 | Loss 0.8957 | Rec 0.8493 | KLD 0.0463
checkpoint saved ✅
epoch=  8
ladder_vae | Epoch 8 | Loss 0.8934 | Rec 0.8451 | KLD 0.0484
checkpoint saved ✅
epoch=  9
ladder_vae | Epoch 9 | Loss 0.8913 | Rec 0.8412 | KLD 0.0500
checkpoint saved ✅
epoch=  10
ladder_vae | Epoch 10 | Loss 0.8898 | Rec 0.8383 | KLD 0.0515
checkpoint saved ✅
epoch=  

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

########################################
# CPU Optimization
########################################

os.environ["OMP_NUM_THREADS"] = "8"
os.environ["MKL_NUM_THREADS"] = "8"

torch.set_num_threads(8)
torch.set_num_interop_threads(8)

device = torch.device("cpu")

class MelDataset(Dataset):

    def __init__(self, root):
        self.files = [
            os.path.join(root, f)
            for f in os.listdir(root)
            if f.endswith(".pt")
        ]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        x = torch.load(self.files[idx], weights_only=True)

        # force fixed size 80x80
        if x.shape[2] > 80:
            x = x[:, :, :80]
        else:
            pad = 80 - x.shape[2]
            x = F.pad(x, (0, pad))

        return x


train_dataset = MelDataset("data/librispeech_mel/train-clean-100")
test_dataset  = MelDataset("data/librispeech_mel/dev-clean")

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=6
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=6
)

In [5]:
# گام ۱: پیاده‌سازی مدل‌های Baseline
import torch
import torch.nn as nn

class BaseVAE(nn.Module):

    def __init__(self, latent_dim=120, input_shape=(1,128,128)):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1,32,3,stride=2,padding=1), nn.ReLU(),
            nn.Conv2d(32,64,3,stride=2,padding=1), nn.ReLU(),
            nn.Conv2d(64,128,3,stride=2,padding=1), nn.ReLU(),
            nn.Conv2d(128,256,3,stride=2,padding=1), nn.ReLU()
        )

        # محاسبه خودکار سایز خروجی encoder
        with torch.no_grad():
            dummy = torch.zeros(1,*input_shape)
            h = self.encoder(dummy)
            self.flatten_dim = h.view(1,-1).shape[1]

        self.fc_mu = nn.Linear(self.flatten_dim, latent_dim)
        self.fc_var = nn.Linear(self.flatten_dim, latent_dim)

        self.decoder_input = nn.Linear(latent_dim, self.flatten_dim)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256,128,3,stride=2,padding=1,output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128,64,3,stride=2,padding=1,output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64,32,3,stride=2,padding=1,output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,1,3,stride=2,padding=1,output_padding=1),
            nn.Sigmoid()
        )


    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self,x):
    
        h = self.encoder(x)
    
        h_flat = h.view(x.size(0),-1)
    
        mu = self.fc_mu(h_flat)
        logvar = self.fc_var(h_flat)
    
        z = self.reparameterize(mu,logvar)
    
        h_dec = self.decoder_input(z)
    
        h_dec = h_dec.view(x.size(0),256, h.shape[2], h.shape[3])
    
        recon = self.decoder(h_dec)
    
        return recon, mu, logvar


#########################################################

######################
class HierarchicalVAE(nn.Module):

    def __init__(self, latent_dim=120, input_shape=(1,80,80)):
        super().__init__()

        self.dims = [40,40,40]

        self.enc1 = nn.Conv2d(1,32,3,stride=2,padding=1)
        self.enc2 = nn.Conv2d(32,64,3,stride=2,padding=1)
        self.enc3 = nn.Conv2d(64,128,3,stride=2,padding=1)

        # ابعاد صحیح برای 80x80
        self.mu_list = nn.ModuleList([
            nn.Linear(32*40*40,40),
            nn.Linear(64*20*20,40),
            nn.Linear(128*10*10,40)
        ])

        self.var_list = nn.ModuleList([
            nn.Linear(32*40*40,40),
            nn.Linear(64*20*20,40),
            nn.Linear(128*10*10,40)
        ])

        self.decoder = nn.Sequential(

            nn.Linear(120,128*10*10),
            nn.ReLU(),

            nn.Unflatten(1,(128,10,10)),

            nn.ConvTranspose2d(128,64,3,2,1,1),
            nn.ReLU(),

            nn.ConvTranspose2d(64,32,3,2,1,1),
            nn.ReLU(),

            nn.ConvTranspose2d(32,1,3,2,1,1),
            nn.Sigmoid()
        )

    def reparameterize(self, mu, logvar):

        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)

        return mu + eps*std


    def forward(self,x):

        x1 = torch.relu(self.enc1(x))
        x2 = torch.relu(self.enc2(x1))
        x3 = torch.relu(self.enc3(x2))

        mu1 = self.mu_list[0](x1.view(x.size(0),-1))
        mu2 = self.mu_list[1](x2.view(x.size(0),-1))
        mu3 = self.mu_list[2](x3.view(x.size(0),-1))

        logvar1 = self.var_list[0](x1.view(x.size(0),-1))
        logvar2 = self.var_list[1](x2.view(x.size(0),-1))
        logvar3 = self.var_list[2](x3.view(x.size(0),-1))

        mu = torch.cat([mu1,mu2,mu3],dim=1)
        logvar = torch.cat([logvar1,logvar2,logvar3],dim=1)

        z = self.reparameterize(mu,logvar)

        recon = self.decoder(z)

        return recon, mu, logvar

##########################


In [6]:
########################################
# Hierarchical Encoder
########################################

class HierarchicalEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(1,32,3,stride=2,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(32,64,3,stride=2,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(64,128,3,stride=2,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(128,128,3,stride=2,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.projections = nn.ModuleList([
            nn.Linear(32,20),
            nn.Linear(64,20),
            nn.Linear(128,20),
            nn.Linear(128,20),
            nn.Linear(128,20),
            nn.Linear(128,20)
        ])

    def forward(self,x):

        f1 = self.conv1(x)
        f2 = self.conv2(f1)
        f3 = self.conv3(f2)
        f4 = self.conv4(f3)

        pooled = [
            self.pool(f1).view(x.size(0),-1),
            self.pool(f2).view(x.size(0),-1),
            self.pool(f3).view(x.size(0),-1),
            self.pool(f4).view(x.size(0),-1),
            self.pool(f4).view(x.size(0),-1),
            self.pool(f4).view(x.size(0),-1),
        ]

        z_levels = [
            self.projections[i](pooled[i])
            for i in range(6)
        ]

        z = torch.cat(z_levels, dim=1)

        return z, z_levels


########################################
# Decoder
########################################

class Decoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.fc = nn.Linear(120,128*5*5)

        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128,128,4,stride=2,padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(128,64,4,stride=2,padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(64,32,4,stride=2,padding=1),
            nn.ReLU(),

            nn.ConvTranspose2d(32,1,4,stride=2,padding=1)
        )

    def forward(self,z):

        x = self.fc(z)
        x = x.view(-1,128,5,5)
        x = self.deconv(x)

        return x


########################################
# Model
########################################

class HDIModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = HierarchicalEncoder()
        self.decoder = Decoder()

    def forward(self,x):

        z, levels = self.encoder(x)
        recon = self.decoder(z)

        return recon, levels

In [3]:
import torch
import numpy as np


def extract_latents_basevae(model, dataloader, device, save_path):
    model.eval()
    model.to(device)

    latents = []

    with torch.no_grad():
        for x in dataloader:
            x = x.to(device)

            h = model.encoder(x)
            h_flat = h.view(x.size(0), -1)

            mu = model.fc_mu(h_flat)
            logvar = model.fc_var(h_flat)

            # برای evaluation معمولا از mu استفاده می‌کنیم
            z = mu

            latents.append(z.cpu().numpy())

    latents = np.concatenate(latents, axis=0)

    np.save(save_path, latents)

    print("saved:", save_path)
    print("shape:", latents.shape)


In [7]:
# اجرای درست برای Vanilla VAE
vanilla_vae = BaseVAE(
    latent_dim=120,
    input_shape=(1, 80, 80)
).to(device)

ckpt = torch.load("vanilla_ae_best.pt", map_location=device, weights_only=False)

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    vanilla_vae.load_state_dict(ckpt["model_state_dict"])
else:
    vanilla_vae.load_state_dict(ckpt)

extract_latents_basevae(
    vanilla_vae,
    test_loader,
    device,
    "latent_vanilla.npy"
)


saved: latent_vanilla.npy
shape: (2703, 120)


In [6]:
beta_vae = BaseVAE(
    latent_dim=120,
    input_shape=(1, 80, 80)
).to(device)

ckpt = torch.load("beta_vae_best.pt", map_location=device,weights_only=False)

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    beta_vae.load_state_dict(ckpt["model_state_dict"])
else:
    beta_vae.load_state_dict(ckpt)

extract_latents_basevae(
    beta_vae,
    test_loader,
    device,
    "latent_beta.npy"
)


saved: latent_beta.npy
shape: (2703, 120)


In [14]:
def extract_latents_hvae(model, dataloader, device, save_path):
    model.eval()
    model.to(device)

    latents = []

    with torch.no_grad():
        for x in dataloader:
            x = x.to(device)

            _, mu, _ = model(x)

            latents.append(mu.cpu().numpy())

    latents = np.concatenate(latents, axis=0)
    np.save(save_path, latents)

    print("saved:", save_path)
    print("shape:", latents.shape)
################


def extract_latents_factorvae(model, dataloader, device, save_path):
    model.eval()
    model.to(device)

    latents = []

    with torch.no_grad():
        for x in dataloader:
            x = x.to(device)

            mu, _ = model.encode(x)

            latents.append(mu.cpu().numpy())

    latents = np.concatenate(latents, axis=0)
    np.save(save_path, latents)

    print("saved:", save_path)
    print("shape:", latents.shape)


############
def extract_latents_ladder(model, dataloader, device, save_path):
    model.eval()
    model.to(device)

    latents = []

    with torch.no_grad():
        for x in dataloader:
            x = x.to(device)

            _, mu, _ = model(x)

            latents.append(mu.cpu().numpy())

    latents = np.concatenate(latents, axis=0)
    np.save(save_path, latents)

    print("saved:", save_path)
    print("shape:", latents.shape)


In [15]:
hvae = HierarchicalVAE().to(device)

ckpt = torch.load("hvae_best.pt", map_location=device,weights_only=False)

if "model_state_dict" in ckpt:
    hvae.load_state_dict(ckpt["model_state_dict"])
else:
    hvae.load_state_dict(ckpt)

extract_latents_hvae(
    hvae,
    test_loader,
    device,
    "latent_hvae.npy"
)


saved: latent_hvae.npy
shape: (2703, 120)


In [17]:
factorvae = FactorVAE(
    latent_dim=120,
    input_shape=(1,80,80)
).to(device)

ckpt = torch.load("factorvae_best.pt", map_location=device,weights_only=False)

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    factorvae.load_state_dict(ckpt["model_state_dict"])
else:
    factorvae.load_state_dict(ckpt)

extract_latents_factorvae(
    factorvae,
    test_loader,
    device,
    "latent_factorvae.npy"
)


saved: latent_factorvae.npy
shape: (2703, 120)


In [19]:
laddervae = LadderVAE(
    latent_dims=[40,40,40]
).to(device)

ckpt = torch.load("ladder_vae_best.pt", map_location=device,weights_only=False)

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    laddervae.load_state_dict(ckpt["model_state_dict"])
else:
    laddervae.load_state_dict(ckpt)

extract_latents_ladder(
    laddervae,
    test_loader,
    device,
    "latent_ladder.npy"
)


saved: latent_ladder.npy
shape: (2703, 120)


In [21]:
import torch
import numpy as np


def extract_latents_hdi(model, dataloader, device, save_prefix="latent_hdi"):

    model.eval()
    model.to(device)

    all_latents = []

    level_latents = [
        [], [], [], [], [], []
    ]

    with torch.no_grad():

        for x in dataloader:

            x = x.to(device)

            # HDI forward:
            # recon, levels
            recon, levels = model(x)

            # levels is list of 6 tensors, each [B, 20]
            z_all = torch.cat(levels, dim=1)   # [B, 120]

            all_latents.append(z_all.cpu().numpy())

            for i in range(6):
                level_latents[i].append(levels[i].cpu().numpy())

    # concatenate all batches
    all_latents = np.concatenate(all_latents, axis=0)

    level_latents = [
        np.concatenate(level_latents[i], axis=0)
        for i in range(6)
    ]

    # save full latent for fair comparison with other baselines
    np.save(f"{save_prefix}_all.npy", all_latents)

    # save each hierarchical level separately
    for i in range(6):
        np.save(f"{save_prefix}_level{i+1}.npy", level_latents[i])

    print("saved:", f"{save_prefix}_all.npy")
    print("shape all:", all_latents.shape)

    for i in range(6):
        print(f"saved: {save_prefix}_level{i+1}.npy")
        print(f"shape level {i+1}:", level_latents[i].shape)

    return all_latents, level_latents


In [22]:

hdi_model = HDIModel().to(device)

ckpt = torch.load("hdi_model_best.pt", map_location=device,weights_only=False)

if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    hdi_model.load_state_dict(ckpt["model_state_dict"])
else:
    hdi_model.load_state_dict(ckpt)

all_latents_hdi, level_latents_hdi = extract_latents_hdi(
    model=hdi_model,
    dataloader=test_loader,
    device=device,
    save_prefix="latent_hdi"
)


saved: latent_hdi_all.npy
shape all: (2703, 120)
saved: latent_hdi_level1.npy
shape level 1: (2703, 20)
saved: latent_hdi_level2.npy
shape level 2: (2703, 20)
saved: latent_hdi_level3.npy
shape level 3: (2703, 20)
saved: latent_hdi_level4.npy
shape level 4: (2703, 20)
saved: latent_hdi_level5.npy
shape level 5: (2703, 20)
saved: latent_hdi_level6.npy
shape level 6: (2703, 20)


In [ ]:
###################################

In [29]:
def compute_level_factor_mi(level_latents, factors, n_bins=10):
    """
    level_latents: list of arrays
        each level shape [N, d_l]
    factors: shape [N, K]

    returns:
        level_factor_mi: shape [L, K]
    """

    L = len(level_latents)
    K = factors.shape[1]

    level_factor_mi = np.zeros((L, K))

    for l in range(L):

        z_l = level_latents[l]
        z_l_disc = discretize_latents(z_l, n_bins=n_bins)

        for k in range(K):

            factor_k = factors[:, k]

            mi_dims = []

            for d in range(z_l_disc.shape[1]):
                mi = mutual_info_score(
                    z_l_disc[:, d],
                    factor_k
                )
                mi_dims.append(mi)

            # میانگین MI تمام ابعاد آن level با factor k
            level_factor_mi[l, k] = np.mean(mi_dims)

    return level_factor_mi


In [30]:
def compute_hds(level_latents, factors, n_bins=10):
    """
    Computes Hierarchical Disentanglement Score.

    level_latents: list of 6 arrays, each [N,20]
    factors: [N,K]
    """

    level_factor_mi = compute_level_factor_mi(
        level_latents,
        factors,
        n_bins=n_bins
    )

    L, K = level_factor_mi.shape

    hds_per_factor = []

    for k in range(K):

        mi_levels = level_factor_mi[:, k]
        sorted_mi = np.sort(mi_levels)[::-1]

        h_k = discrete_entropy(factors[:, k])

        if h_k < 1e-12 or len(sorted_mi) < 2:
            hds_k = 0.0
        else:
            hds_k = (sorted_mi[0] - sorted_mi[1]) / h_k

        hds_per_factor.append(hds_k)

    hds = float(np.mean(hds_per_factor))

    return hds, hds_per_factor, level_factor_mi


In [31]:
# تابع نهایی evaluation همه مدل‌ها
def evaluate_all_models(latent_files, factors):
    """
    latent_files: dict
        {
            "Vanilla VAE": "latent_vanilla.npy",
            ...
        }

    factors: np.array [N,K]
    """

    rows = []

    for model_name, latent_path in latent_files.items():

        print("=" * 60)
        print("Evaluating:", model_name)
        print("Loading:", latent_path)

        z = np.load(latent_path)

        print("latent shape:", z.shape)

        mig, mig_per_factor, mi_matrix = compute_mig(
            z,
            factors,
            n_bins=20
        )

        factor_score, factor_accs = compute_factorvae_score(
            z,
            factors
        )

        rows.append({
            "Model": model_name,
            "MIG": mig,
            "FactorVAE_Score": factor_score,
            "Latent_Dim": z.shape[1]
        })

        print("MIG:", mig)
        print("FactorVAE Score:", factor_score)

    results_df = pd.DataFrame(rows)

    return results_df


In [32]:
latent_files = {
    "Vanilla VAE": "latent_vanilla.npy",
    "Beta-VAE": "latent_beta.npy",
    "FactorVAE": "latent_factorvae.npy",
    "HVAE": "latent_hvae.npy",
    "LadderVAE": "latent_ladder.npy",
    "HDI": "latent_hdi_all.npy",
}


In [ ]:
########### اسکریپت ساخت factors

In [47]:
# ================================
# Imports
# ================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import mutual_info_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


# ================================
# Discretize latents
# ================================

def discretize_latents(z, n_bins=10):
    """
    z: shape [N, D]
    return: discretized z with shape [N, D]
    """

    discretizer = KBinsDiscretizer(
        n_bins=n_bins,
        encode="ordinal",
        strategy="quantile"
    )

    z_disc = discretizer.fit_transform(z)

    return z_disc.astype(int)


# ================================
# Entropy
# ================================

def discrete_entropy(y):
    """
    y: shape [N]
    """

    values, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()

    entropy = -np.sum(probs * np.log(probs + 1e-12))

    return entropy


# ================================
# MIG
# ================================

def compute_mig(z, factors, n_bins=20):
    """
    z: shape [N, D]
    factors: shape [N, K]
    """

    z_disc = discretize_latents(z, n_bins=n_bins)

    N, D = z_disc.shape
    K = factors.shape[1]

    mig_scores = []
    mi_matrix = np.zeros((D, K))

    for k in range(K):

        factor_k = factors[:, k]

        for d in range(D):

            mi_matrix[d, k] = mutual_info_score(
                z_disc[:, d],
                factor_k
            )

        sorted_mi = np.sort(mi_matrix[:, k])[::-1]

        if len(sorted_mi) < 2:
            mig_k = 0.0
        else:
            h_k = discrete_entropy(factor_k)

            if h_k < 1e-12:
                mig_k = 0.0
            else:
                mig_k = (sorted_mi[0] - sorted_mi[1]) / h_k

        mig_scores.append(mig_k)

    mig = float(np.mean(mig_scores))

    return mig, mig_scores, mi_matrix


# ================================
# FactorVAE score
# ================================

def compute_factorvae_score(z, factors, test_size=0.3, random_state=42):
    """
    Practical FactorVAE-style score using factor prediction accuracy.
    """

    K = factors.shape[1]

    acc_scores = []

    for k in range(K):

        y = factors[:, k]

        if len(np.unique(y)) < 2:
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            z,
            y,
            test_size=test_size,
            random_state=random_state,
            stratify=y
        )

        clf = RandomForestClassifier(
            n_estimators=200,
            random_state=random_state,
            n_jobs=-1
        )

        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)

        acc = accuracy_score(y_test, y_pred)

        acc_scores.append(acc)

    if len(acc_scores) == 0:
        return 0.0, []

    score = float(np.mean(acc_scores))

    return score, acc_scores


# ================================
# Level-Factor MI
# ================================

def compute_level_factor_mi(level_latents, factors, n_bins=10):

    L = len(level_latents)
    K = factors.shape[1]

    level_factor_mi = np.zeros((L, K))

    for l in range(L):

        z_l = level_latents[l]
        z_l_disc = discretize_latents(z_l, n_bins=n_bins)

        for k in range(K):

            factor_k = factors[:, k]

            mi_dims = []

            for d in range(z_l_disc.shape[1]):

                mi = mutual_info_score(
                    z_l_disc[:, d],
                    factor_k
                )

                mi_dims.append(mi)

            level_factor_mi[l, k] = np.mean(mi_dims)

    return level_factor_mi


# ================================
# HDS
# ================================

def compute_hds(level_latents, factors, n_bins=10):

    level_factor_mi = compute_level_factor_mi(
        level_latents,
        factors,
        n_bins=n_bins
    )

    L, K = level_factor_mi.shape

    hds_per_factor = []

    for k in range(K):

        mi_levels = level_factor_mi[:, k]
        sorted_mi = np.sort(mi_levels)[::-1]

        h_k = discrete_entropy(factors[:, k])

        if h_k < 1e-12 or len(sorted_mi) < 2:
            hds_k = 0.0
        else:
            hds_k = (sorted_mi[0] - sorted_mi[1]) / h_k

        hds_per_factor.append(hds_k)

    hds = float(np.mean(hds_per_factor))

    return hds, hds_per_factor, level_factor_mi


# ================================
# Evaluate all models
# ================================

def evaluate_all_models(latent_files, factors):

    rows = []

    for model_name, latent_path in latent_files.items():

        print("=" * 60)
        print("Evaluating:", model_name)

        z = np.load(latent_path)

        print("latent shape:", z.shape)

        mig, mig_per_factor, mi_matrix = compute_mig(
            z,
            factors,
            n_bins=20
        )

        factor_score, factor_accs = compute_factorvae_score(
            z,
            factors
        )

        rows.append({
            "Model": model_name,
            "MIG": mig,
            "FactorVAE_Score": factor_score,
            "Latent_Dim": z.shape[1]
        })

        print("MIG:", mig)
        print("FactorVAE Score:", factor_score)

    results_df = pd.DataFrame(rows)

    return results_df


# ================================
# Latent files
# ================================

latent_files = {
    "Vanilla VAE": "latent_vanilla.npy",
    "Beta-VAE": "latent_beta.npy",
    "FactorVAE": "latent_factorvae.npy",
    "HVAE": "latent_hvae.npy",
    "LadderVAE": "latent_ladder.npy",
    "HDI": "latent_hdi_all.npy",
}




In [48]:
# ================================
# Run evaluation
# ================================

factors = np.load("factors_dev_clean.npy")

results_df = evaluate_all_models(
    latent_files,
    factors
)

print("\nFinal Results\n")
print(results_df)

results_df.to_csv(
    "disentanglement_results.csv",
    index=False
)

print("\nSaved: disentanglement_results.csv")


Evaluating: Vanilla VAE
latent shape: (2703, 120)
MIG: 0.00138165710112589
FactorVAE Score: 0.43107274969173853
Evaluating: Beta-VAE
latent shape: (2703, 120)
MIG: 0.0010231781259472197
FactorVAE Score: 0.41085080147965475
Evaluating: FactorVAE
latent shape: (2703, 120)
MIG: 0.0014321638697932103
FactorVAE Score: 0.41454993834771886
Evaluating: HVAE
latent shape: (2703, 120)
MIG: 0.0005000357641906726
FactorVAE Score: 0.42219482120838475
Evaluating: LadderVAE
latent shape: (2703, 120)
MIG: 0.0007500762764348161
FactorVAE Score: 0.4241676942046856
Evaluating: HDI
latent shape: (2703, 120)
MIG: 0.00036332109703714714
FactorVAE Score: 0.42441430332922314

Final Results

         Model       MIG  FactorVAE_Score  Latent_Dim
0  Vanilla VAE  0.001382         0.431073         120
1     Beta-VAE  0.001023         0.410851         120
2    FactorVAE  0.001432         0.414550         120
3         HVAE  0.000500         0.422195         120
4    LadderVAE  0.000750         0.424168         120


In [46]:
import numpy as np
import pandas as pd


# -----------------------------
# split latent into hierarchy
# -----------------------------

def split_levels(z, n_levels=6):

    latent_dim = z.shape[1]
    level_dim = latent_dim // n_levels

    levels = []

    for i in range(n_levels):

        start = i * level_dim
        end = (i + 1) * level_dim

        levels.append(z[:, start:end])

    return levels


# -----------------------------
# compute HDS for one model
# -----------------------------

def compute_hds_for_model(latent_file, factors):

    print("Loading:", latent_file)

    z = np.load(latent_file)

    print("latent shape:", z.shape)

    level_latents = split_levels(z, n_levels=6)

    hds, hds_per_factor, level_factor_mi = compute_hds(
        level_latents,
        factors,
        n_bins=10
    )

    return hds, hds_per_factor, level_factor_mi


# -----------------------------
# models
# -----------------------------

hierarchical_models = {
    "HVAE": "latent_hvae.npy",
    "LadderVAE": "latent_ladder.npy",
    "HDI": "latent_hdi_all.npy"
}


# -----------------------------
# load factors
# -----------------------------

factors = np.load("factors_dev_clean.npy")


# -----------------------------
# run evaluation
# -----------------------------

rows = []

for name, path in hierarchical_models.items():

    print("\n" + "="*50)
    print("Evaluating HDS:", name)

    hds, hds_per_factor, level_factor_mi = compute_hds_for_model(
        path,
        factors
    )

    rows.append({
        "Model": name,
        "HDS": hds
    })

    print("HDS:", hds)


hds_results = pd.DataFrame(rows)

print("\nHDS Results\n")
print(hds_results)

hds_results.to_csv(
    "hierarchical_disentanglement_results.csv",
    index=False
)

print("\nSaved: hierarchical_disentanglement_results.csv")



Evaluating HDS: HVAE
Loading: latent_hvae.npy
latent shape: (2703, 120)
HDS: 0.0003444031340657227

Evaluating HDS: LadderVAE
Loading: latent_ladder.npy
latent shape: (2703, 120)
HDS: 0.0001051212039735692

Evaluating HDS: HDI
Loading: latent_hdi_all.npy
latent shape: (2703, 120)
HDS: 0.00047709922223751494

HDS Results

       Model       HDS
0       HVAE  0.000344
1  LadderVAE  0.000105
2        HDI  0.000477

Saved: hierarchical_disentanglement_results.csv
